In [5]:
DATADIR = "/Users/katst103/Documents/GitHub/comparativeCD/data/05_network/cCRE/" 

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import seaborn

AttributeError: partially initialized module 'pandas' has no attribute '_pandas_datetime_CAPI' (most likely due to a circular import)

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.gridspec as gridspec

# ── Load data ──────────────────────────────────────────────────────────────────
probs = pd.read_csv(DATADIR + 'emissions_state_probabilities.txt', sep='\t')
annot = pd.read_csv(DATADIR + 'emissions_genome_annotations_enrichment.txt', sep='\t')

# Keep only presence=1 rows (probability that mark IS present)
probs1 = probs[probs['presence'] == 1].copy()
marks_order = ['ATAC', 'H3K27ac', 'H3K4me3', 'H3K27me3']
probs1['mark'] = pd.Categorical(probs1['mark'], categories=marks_order, ordered=True)
probs_wide = probs1.pivot(index='state', columns='mark', values='prob')[marks_order]

# Annotation enrichment matrix (drop Genome % column for heatmap)
annot = annot.set_index('State (Emission order)')
annot_cols = ['RefSeqTSS.cf4.bed.gz', 'RefSeqTSS2kb.cf4.bed.gz', 'RefSeqExon.cf4.bed.gz', 'RefSeqTES.cf4.bed.gz', 'RefSeqGene.cf4.bed.gz']
annot_mat = annot[annot_cols]

# ── State annotations ──────────────────────────────────────────────────────────
state_labels = {
    1: 'Promoter',
    2: 'Enhancer',
    3: 'Open chromatin',
    4: 'Promoter',
    5: 'Promoter',
    6: 'Promoter',
    7: 'No signal',
    8: 'Enhancer',
    9: 'Repressed',
}
state_colors = {
    'Promoter':        '#F4A7B9',
    'Enhancer':        '#4A90D9',
    'Open chromatin':  '#C8860A',
    'No signal':       '#FFFFFF',
    'Repressed':       '#5A8A3C',
}

n_states = 9

# ── Color maps ─────────────────────────────────────────────────────────────────
white_blue = LinearSegmentedColormap.from_list('wb', ['white', '#1A3A8F'])
white_blue_annot = LinearSegmentedColormap.from_list('wb2', ['#EEF2FF', '#1A3A8F'])

# ── Figure layout ──────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 8))
gs = gridspec.GridSpec(
    1, 4,
    width_ratios=[len(marks_order), len(annot_cols), 0.6, 2.2],
    wspace=0.08,
    left=0.06, right=0.98, top=0.93, bottom=0.18
)

ax_emit  = fig.add_subplot(gs[0])   # emission probabilities
ax_annot = fig.add_subplot(gs[1])   # genome annotation enrichment
ax_label = fig.add_subplot(gs[2])   # state label bars
ax_leg   = fig.add_subplot(gs[3])   # legend (invisible axes)

# ── Panel 1: Emission probabilities ───────────────────────────────────────────
im1 = ax_emit.imshow(
    probs_wide.values,
    aspect='auto', cmap=white_blue, vmin=0, vmax=1,
    interpolation='nearest'
)
ax_emit.set_xticks(range(len(marks_order)))
ax_emit.set_xticklabels(marks_order, rotation=45, ha='right', fontsize=10)
ax_emit.set_yticks(range(n_states))
ax_emit.set_yticklabels([str(i) for i in range(1, n_states + 1)], fontsize=10)
ax_emit.set_ylabel('State (Emission order)', fontsize=11, labelpad=6)
ax_emit.set_xlabel('Mark', fontsize=11)
ax_emit.tick_params(top=False, right=False)

# grid lines
for x in np.arange(-0.5, len(marks_order), 1):
    ax_emit.axvline(x, color='white', lw=1.5)
for y in np.arange(-0.5, n_states, 1):
    ax_emit.axhline(y, color='white', lw=1.5)

# ── Panel 2: Annotation enrichment ────────────────────────────────────────────
vmax_annot = annot_mat.values.max()
im2 = ax_annot.imshow(
    annot_mat.values,
    aspect='auto', cmap=white_blue_annot, vmin=0, vmax=vmax_annot,
    interpolation='nearest'
)
col_labels_clean = ['RefSeqTSS', 'RefSeqTSS2kb', 'RefSeqExon', 'RefSeqTES', 'RefSeqGene']
ax_annot.set_xticks(range(len(annot_cols)))
ax_annot.set_xticklabels(col_labels_clean, rotation=45, ha='right', fontsize=9)
ax_annot.set_yticks([])
ax_annot.tick_params(top=False, right=False)
ax_annot.set_xlabel('Genome annotation', fontsize=11)

for x in np.arange(-0.5, len(annot_cols), 1):
    ax_annot.axvline(x, color='white', lw=1.5)
for y in np.arange(-0.5, n_states, 1):
    ax_annot.axhline(y, color='white', lw=1.5)

# ── Panel 3: State label bars ──────────────────────────────────────────────────
ax_label.set_xlim(0, 1)
ax_label.set_ylim(-0.5, n_states - 0.5)
ax_label.invert_yaxis()
ax_label.axis('off')

for i, state in enumerate(range(1, n_states + 1)):
    label = state_labels[state]
    color = state_colors[label]
    edge  = 'none' if label != 'No signal' else '#AAAAAA'
    rect  = mpatches.FancyBboxPatch(
        (0.05, i - 0.38), 0.9, 0.76,
        boxstyle='round,pad=0.02',
        facecolor=color,
        edgecolor=edge, linewidth=0.8
    )
    ax_label.add_patch(rect)
    txt_color = 'white' if label in ('Enhancer', 'Repressed', 'Open chromatin') else 'black'
    ax_label.text(
        0.5, i, label,
        ha='center', va='center',
        fontsize=9, fontweight='bold', color=txt_color
    )

# ── Panel 4: Legend (colorbars) ────────────────────────────────────────────────
ax_leg.axis('off')

# Emission probability colorbar
cax1 = fig.add_axes([0.77, 0.55, 0.015, 0.30])
cb1  = fig.colorbar(im1, cax=cax1)
cb1.set_label('Emission\nprobability', fontsize=9)
cb1.ax.tick_params(labelsize=8)

# Annotation enrichment colorbar
cax2 = fig.add_axes([0.77, 0.18, 0.015, 0.30])
cb2  = fig.colorbar(im2, cax=cax2)
cb2.set_label('Fold\nenrichment', fontsize=9)
cb2.ax.tick_params(labelsize=8)

# State category legend
legend_items = [
    ('Promoter',       '#F4A7B9', 'black'),
    ('Enhancer',       '#4A90D9', 'white'),
    ('Open chromatin', '#C8860A', 'white'),
    ('No signal',      '#FFFFFF', 'black'),
    ('Repressed',      '#5A8A3C', 'white'),
]
legend_x, legend_y0 = 0.82, 0.88
for i, (lbl, col, tc) in enumerate(legend_items):
    y = legend_y0 - i * 0.06
    rect = mpatches.FancyBboxPatch(
        (legend_x, y - 0.02), 0.12, 0.04,
        boxstyle='round,pad=0.005',
        facecolor=col,
        edgecolor='#AAAAAA' if col == '#FFFFFF' else 'none',
        linewidth=0.8,
        transform=fig.transFigure, clip_on=False
    )
    fig.add_artist(rect)
    fig.text(legend_x + 0.14, y, lbl, va='center', fontsize=9,
             transform=fig.transFigure)

fig.text(legend_x, legend_y0 + 0.05, 'State annotation',
         fontsize=10, fontweight='bold', transform=fig.transFigure)

# ── Title ──────────────────────────────────────────────────────────────────────
fig.suptitle('Chromatin state emission probabilities and genomic enrichment',
             fontsize=12, fontweight='bold', y=0.98)

plt.savefig('/mnt/user-data/outputs/emission_states_plot.pdf',
            format='pdf', bbox_inches='tight', dpi=300)
plt.savefig('/mnt/user-data/outputs/emission_states_plot.png',
            format='png', bbox_inches='tight', dpi=300)
print("Done")

AttributeError: partially initialized module 'pandas' has no attribute '_pandas_datetime_CAPI' (most likely due to a circular import)